# Notebook 33 — Gemma 4: checks that need transformers 5

**Author:** Sunita Sahu · **Decision:** D9 (roster extension) · CPU only · **environment: transformers ≥ 5.10** (not the 4.57 environment of notebooks 26–32)

Gemma 4 12B (`Gemma4UnifiedForConditionalGeneration`, encoder-free) and Gemma 4 26B-A4B (`Gemma4ForConditionalGeneration`, vision encoder + mixture-of-experts language model) exist only in transformers ≥ 5.10 / 5.5. Notebooks 26–28 and 32 run in transformers 4.57 and use, for Gemma 4, the checkpoint index (parameter shapes) and a transcription of the released image preprocessing (token counts). This notebook checks both against the released code, validates the cost model, and counts the operator calls of one decode step for notebook 31 (decision log N28).

**Set-up.** A separate virtual environment: `pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu` and `pip install transformers==5.18.0 pillow sentencepiece protobuf safetensors pandas scipy`. Run from `code_active/notebooks/`. Outputs: `outputs/measurements/nb33_*.csv`.

---
## Setup

In [1]:
import sys, time
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
CODE_ACTIVE = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
sys.path.insert(0, str(CODE_ACTIVE))
MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
LOG_DIR = CODE_ACTIVE / 'outputs' / 'logs'

import transformers, torch
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()
major, minor = (int(part) for part in transformers.__version__.split('.')[:2])
assert (major, minor) >= (5, 10), f'transformers {transformers.__version__}: Gemma 4 12B needs >= 5.10'
from vlm_audit.logging_utils import ExperimentLogger, SessionTranscript
from vlm_audit.token_accounting import account_gemma4, count_image_tokens, load_processor, synthetic_image
from vlm_audit.flop_counter_validation import TRANSFORMERS5_VALIDATION_SPECS, measure_elementwise_share, validate_model
if globals().get('transcript') is not None:
    transcript.close()
transcript = SessionTranscript(LOG_DIR, '33_transformers5_validation')
logger = ExperimentLogger(LOG_DIR, '33_transformers5_validation')
GEMMA_MODELS = list(TRANSFORMERS5_VALIDATION_SPECS)
print('transformers', transformers.__version__, '| torch', torch.__version__, '| models', GEMMA_MODELS)

<home>\AppData\Local\Temp\claude\<session>\0e631d38-01d3-4ff7-b6de-e525afd6207e\scratchpad\venv_t5\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Session transcript: <repository>\code_active\outputs\logs\20261004_125637_33_transformers5_validation__session_transcript.txt
Crash traces:       <repository>\code_active\outputs\logs\20261004_125637_33_transformers5_validation__crash_traces.txt
transformers 5.18.0 | torch 2.14.1+cpu | models ['Gemma-4-12B', 'Gemma-4-26B-A4B']


---
## 1. Token accounting against the released processors

For every image shape: the real (unpadded) patches, the image tokens in the language sequence (soft tokens plus the begin/end markers) and the padded patch slots the reference implementation runs on.

In [2]:
SHAPES = sorted({(round(side * height_ratio), round(side * width_ratio))
                 for side in (100, 224, 448, 640, 1024, 2048, 4096)
                 for height_ratio, width_ratio in ((1, 1), (1, .5), (.5, 1), (1, .25), (.25, 1), (1, .75), (.75, 1),
                                                   (1, 1 / 3), (.1, 1))})
accounting_rows = []
for name in GEMMA_MODELS:
    processor = load_processor(name)
    for height, width in SHAPES:
        batch = processor(text='<|image|> Describe this image in detail.', images=synthetic_image(height, width),
                          return_tensors='pt')
        intrinsic = account_gemma4(name, height, width)
        executed = account_gemma4(name, height, width, executed=True)
        real_patches = int((batch['image_position_ids'][0] != -1).all(dim=-1).sum())
        expected_patches = intrinsic.patch_positions if name == 'Gemma-4-26B-A4B' else intrinsic.fusion_output_tokens
        accounting_rows.append({
            'model': name, 'height': height, 'width': width,
            'processor_real_patches': real_patches, 'accounting_real_patches': expected_patches,
            'processor_sequence_image_tokens': count_image_tokens(name, batch['input_ids'][0]),
            'accounting_sequence_image_tokens': intrinsic.language_visual_tokens,
            'processor_padded_slots': int(batch['pixel_values'].shape[1]),
            'accounting_padded_slots': executed.patch_positions})
accounting = pd.DataFrame(accounting_rows)
accounting['match'] = ((accounting['processor_real_patches'] == accounting['accounting_real_patches'])
                       & (accounting['processor_sequence_image_tokens'] == accounting['accounting_sequence_image_tokens'])
                       & (accounting['processor_padded_slots'] == accounting['accounting_padded_slots']))
accounting.to_csv(MEASUREMENT_DIR / 'nb33_gemma4_token_accounting_check.csv', index=False)
assert accounting['match'].all(), accounting[~accounting['match']]
logger.info(f'token accounting: {len(accounting)} comparisons, all match')
print(f'{len(accounting)} comparisons ({len(SHAPES)} shapes x {len(GEMMA_MODELS)} models): all match')
accounting.groupby('model')[['processor_sequence_image_tokens', 'processor_padded_slots']].agg(['min', 'max'])

126 comparisons (63 shapes x 2 models): all match


processor_sequence_image_tokens      processor_padded_slots  \
                                            min  max                    min   
model                                                                         
Gemma-4-12B                                 254  268                    280   
Gemma-4-26B-A4B                             254  268                   2520   

                       
                  max  
model                  
Gemma-4-12B       280  
Gemma-4-26B-A4B  2520

---
## 2. Cost model against the op-level FLOP counter

Reduced depth: one sliding-window and one global decoder layer (the configuration is rebuilt from the released `config.json`, so each layer keeps its released head size and key/value heads), two vision-encoder layers for 26B-A4B, random bf16 weights, eager attention and the eager expert loop. The executed token account is used (padded patch slots; the 26B-A4B pooling product).

In [3]:
validation_rows, head_rows = [], []
for name in GEMMA_MODELS:
    started = time.time()
    result = validate_model(name)
    validation_rows.extend(result['rows'])
    head_rows.append({'model': name, 'measured_head_gflops': result['measured_head_gflops'],
                      'predicted_head_gflops': result['predicted_head_gflops'], 'seconds': round(time.time() - started)})
    logger.info(f'{name}: validated in {time.time() - started:.0f} s')
validation = pd.DataFrame(validation_rows)
validation['environment'] = f'transformers {transformers.__version__}'
validation.to_csv(MEASUREMENT_DIR / 'nb33_flop_counter_validation.csv', index=False)
print(f"{len(validation)} component comparisons; max |relative difference| = "
      f"{validation['relative_difference_pct'].abs().max():.4f}%")
pd.concat([validation.round(4), pd.DataFrame(head_rows).round(4)], keys=['components', 'heads'])

6 component comparisons; max |relative difference| = 0.0002%


model component  measured_gflops  predicted_gflops  \
components 0      Gemma-4-12B    vision          14.8636           14.8636   
           1      Gemma-4-12B    fusion           8.2575            8.2575   
           2      Gemma-4-12B  language         249.4533          249.4531   
           3  Gemma-4-26B-A4B    vision         268.0553          268.0551   
           4  Gemma-4-26B-A4B    fusion           1.6609            1.6609   
           5  Gemma-4-26B-A4B  language         117.0449          117.0447   
heads      0      Gemma-4-12B       NaN              NaN               NaN   
           1  Gemma-4-26B-A4B       NaN              NaN               NaN   

              relative_difference_pct          environment  \
components 0                   0.0000  transformers 5.18.0   
           1                   0.0000  transformers 5.18.0   
           2                  -0.0001  transformers 5.18.0   
           3                  -0.0001  transformers 5.18.0   
           4                   0.0000  transformers 5.18.0   
           5                  -0.0002  transformers 5.18.0   
heads      0                      NaN                  NaN   
           1                      NaN                  NaN   

              measured_head_gflops  predicted_head_gflops  seconds  
components 0                   NaN                    NaN      NaN  
           1                   NaN                    NaN      NaN  
           2                   NaN                    NaN      NaN  
           3                   NaN                    NaN      NaN  
           4                   NaN                    NaN      NaN  
           5                   NaN                    NaN      NaN  
heads      0                2.0133                 2.0133     48.0  
           1                1.4764                 1.4764     51.0

---
## 3. Element-wise share of forward FLOPs

In [4]:
elementwise = pd.DataFrame([measure_elementwise_share(name) for name in GEMMA_MODELS]).set_index('model')
elementwise.to_csv(MEASUREMENT_DIR / 'nb33_elementwise_share.csv')
elementwise.round(4)

,matmul_gflops,elementwise_gflops,elementwise_share_pct
model,,,
Gemma-4-12B,274.5877,0.1915,0.0697
Gemma-4-26B-A4B,388.2375,2.0265,0.5193


---
## 4. Operator calls per decode step (decision log N28)

At batch size one the GPU run's decode time follows the number of operator calls, not the weights. `operator_dispatch.count_decode_step` counts the kernel-launching calls of one decode step on the reduced copy, with the library's default attention and expert kernels (as the released checkpoints ran on the GPU: SDPA, and `grouped_mm` for the 26B-A4B experts), and scales them to full depth by layer type (sliding / full attention). Qwen3-VL-8B is counted here as well as in notebook 31 (transformers 4.57.6): the same architecture under two library releases. Notebook 31 reads `nb33_operator_calls.csv` and uses, for each model, the count taken under the release of its GPU measurement.

In [5]:
from vlm_audit.operator_dispatch import BRIDGE_MODEL, TRANSFORMERS5_DECODE_MODELS, decode_step_table
operator_counts = decode_step_table([*TRANSFORMERS5_DECODE_MODELS, BRIDGE_MODEL], logger=logger)
operator_counts.to_csv(MEASUREMENT_DIR / 'nb33_operator_calls.csv', index=False)
operator_counts[['model', 'status', 'transformers_version', 'experts_implementation', 'calls_per_decode_step',
                 'calls_outside_blocks', 'calls_per_block_by_type', 'full_depth_blocks_by_type', 'seconds']]

Gemma-4-12B: 4460 calls per decode step (48 blocks; {"Gemma4UnifiedTextDecoderLayer:full_attention": 88, "Gemma4UnifiedTextDecoderLayer:sliding_attention": 91})


Gemma-4-26B-A4B: 4553 calls per decode step (30 blocks; {"Gemma4TextDecoderLayer:full_attention": 150, "Gemma4TextDecoderLayer:sliding_attention": 149})


Qwen3-VL-8B: 2076 calls per decode step (36 blocks; {"Qwen3VLTextDecoderLayer": 56})


,model,status,transformers_version,experts_implementation,calls_per_decode_step,calls_outside_blocks,calls_per_block_by_type,full_depth_blocks_by_type,seconds
0,Gemma-4-12B,complete,5.18.0,"{'': 'eager', 'text_config': 'eager', 'vision_...",4460,116,"{""Gemma4UnifiedTextDecoderLayer:full_attention...","{""Gemma4UnifiedTextDecoderLayer:full_attention...",43.7
1,Gemma-4-26B-A4B,complete,5.18.0,"{'': 'grouped_mm', 'text_config': 'grouped_mm'...",4553,78,"{""Gemma4TextDecoderLayer:full_attention"": 150,...","{""Gemma4TextDecoderLayer:full_attention"": 5, ""...",49.4
2,Qwen3-VL-8B,complete,5.18.0,"{'': 'eager', 'vision_config': 'eager', 'text_...",2076,60,"{""Qwen3VLTextDecoderLayer"": 56}","{""Qwen3VLTextDecoderLayer"": 36}",38.1


In [6]:
logger.close()
transcript.close()
transcript = None